# Gold Location Trends Mart

**Purpose**: Geographic hiring trends and location-based analytics

**Target Table**: `workspace.gold.gold_location_trends`

**Grain**: Location × Date

**Refresh Strategy**: Full refresh (CREATE OR REPLACE) with metadata logging

**Parameters**:
- `lookback_days`: Number of days of historical data (default: 365)
- `force_full_refresh`: Boolean flag to force table recreation

**Key Metrics**:
- `total_jobs`: Total jobs in location
- `remote_jobs`: Remote job count
- `onsite_jobs`: Onsite job count
- `avg_salary_usd`: Average salary
- `top_sector`: Dominant sector
- `updated_at`: Last refresh timestamp

**Data Sources**:
- `workspace.warehouse.fact_job_postings`
- `workspace.warehouse.dim_location`

**Schema** (per contract):
- Primary key: (location_sk, trend_date_sk)

**Metadata Logging**:
- Tracks run history in `workspace.metadata.gold_location_trends_refresh_log`
- Captures: rows processed, unique locations/dates, processing time, status

In [0]:
# Configuration
target_table = "workspace.gold.gold_location_trends"
metadata_table = "workspace.metadata.gold_location_trends_refresh_log"

# Parameters
lookback_days = 365  # How many days of history to include
force_full_refresh = False  # Set to True to drop and recreate table

In [0]:
import uuid
from datetime import datetime

# Generate unique run ID
run_id = str(uuid.uuid4())
run_timestamp = datetime.now()
status = "RUNNING"
error_message = None

print(f"Run ID: {run_id}")
print(f"Run Timestamp: {run_timestamp}")
print(f"Lookback Days: {lookback_days}")
print(f"Force Full Refresh: {force_full_refresh}")

In [0]:
%sql
-- Create table if it doesn't exist (will use existing if already there)
CREATE TABLE IF NOT EXISTS workspace.metadata.gold_location_trends_refresh_log (
  run_id STRING NOT NULL,
  run_timestamp TIMESTAMP NOT NULL,
  rows_inserted BIGINT,
  status STRING NOT NULL,
  lookback_days INT,
  rows_processed BIGINT,
  unique_locations INT,
  unique_dates INT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DECIMAL(10,2),
  error_message STRING
)
USING DELTA
COMMENT 'Audit log for gold_location_trends refresh operations';

In [0]:
%sql
-- Add new columns to existing audit table
ALTER TABLE workspace.metadata.gold_location_trends_refresh_log
ADD COLUMNS (
  lookback_days INT,
  rows_processed BIGINT,
  unique_locations INT,
  unique_dates INT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DECIMAL(10,2),
  error_message STRING
);

In [0]:
# Optionally drop table for full refresh
if force_full_refresh:
    print(f"Force full refresh enabled - dropping table {target_table}")
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    print("✓ Table dropped")
else:
    print("Proceeding with CREATE OR REPLACE (preserves table properties)")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_location_trends (
  location_sk BIGINT NOT NULL COMMENT 'Location key',
  trend_date_sk INT NOT NULL COMMENT 'Date key',
  total_jobs BIGINT COMMENT 'Total jobs in location',
  remote_jobs BIGINT COMMENT 'Remote job count',
  onsite_jobs BIGINT COMMENT 'Onsite job count',
  avg_salary_usd DECIMAL(15,2) COMMENT 'Average salary',
  top_sector STRING COMMENT 'Dominant sector',
  updated_at TIMESTAMP NOT NULL COMMENT 'Last refresh'
)
USING DELTA
COMMENT 'Geographic hiring trends and location-based analytics'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
%sql
INSERT OVERWRITE TABLE workspace.gold.gold_location_trends

WITH daily_location AS (
  SELECT
    f.location_sk,
    f.posting_date_sk AS trend_date_sk,
    f.sector_sk,
    j.remote_type,
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS total_jobs,
    AVG(CASE WHEN f.active_flag = TRUE THEN (j.salary_min + j.salary_max) / 2 END) AS avg_salary
  FROM workspace.warehouse.fact_job_postings f
  LEFT JOIN workspace.warehouse.dim_job j ON f.job_sk = j.job_sk AND j.is_current = TRUE
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
    AND f.location_sk IS NOT NULL
  GROUP BY f.location_sk, f.posting_date_sk, f.sector_sk, j.remote_type
),

top_sectors AS (
  SELECT
    location_sk,
    trend_date_sk,
    sector_sk,
    SUM(total_jobs) AS sector_jobs,
    ROW_NUMBER() OVER (PARTITION BY location_sk, trend_date_sk ORDER BY SUM(total_jobs) DESC, sector_sk) AS rn
  FROM daily_location
  GROUP BY location_sk, trend_date_sk, sector_sk
),

aggregated AS (
  SELECT
    dl.location_sk,
    dl.trend_date_sk,
    SUM(dl.total_jobs) AS total_jobs,
    SUM(CASE WHEN dl.remote_type = 'REMOTE' THEN dl.total_jobs ELSE 0 END) AS remote_jobs,
    SUM(CASE WHEN dl.remote_type IN ('ONSITE', 'HYBRID') THEN dl.total_jobs ELSE 0 END) AS onsite_jobs,
    AVG(dl.avg_salary) AS avg_salary_usd,
    FIRST(ds.sector_name) AS top_sector
  FROM daily_location dl
  LEFT JOIN top_sectors ts ON dl.location_sk = ts.location_sk AND dl.trend_date_sk = ts.trend_date_sk AND ts.rn = 1
  LEFT JOIN workspace.warehouse.dim_sector ds ON ts.sector_sk = ds.sector_sk
  GROUP BY dl.location_sk, dl.trend_date_sk
)

SELECT
  location_sk,
  trend_date_sk,
  total_jobs,
  remote_jobs,
  onsite_jobs,
  CAST(avg_salary_usd AS DECIMAL(15,2)) AS avg_salary_usd,
  top_sector,
  CURRENT_TIMESTAMP() AS updated_at
FROM aggregated
ORDER BY trend_date_sk DESC, total_jobs DESC;

In [0]:
import time

start_time = time.time()

try:
    # Get summary metrics from the gold table
    metrics = spark.sql(f"""
        SELECT 
            COUNT(*) as rows_processed,
            COUNT(DISTINCT location_sk) as unique_locations,
            COUNT(DISTINCT trend_date_sk) as unique_dates
        FROM {target_table}
    """).collect()[0]
    
    rows_processed = metrics['rows_processed']
    unique_locations = metrics['unique_locations']
    unique_dates = metrics['unique_dates']
    processing_time = time.time() - start_time
    status = "SUCCESS"
    
    print(f"✓ Processed {rows_processed:,} rows")
    print(f"✓ Unique locations: {unique_locations}")
    print(f"✓ Unique dates: {unique_dates}")
    print(f"✓ Processing time: {processing_time:.2f}s")
    
except Exception as e:
    status = "FAILED"
    error_message = str(e)
    processing_time = time.time() - start_time
    rows_processed = 0
    unique_locations = 0
    unique_dates = 0
    print(f"✗ Error: {error_message}")
    raise

finally:
    # Write metadata log
    spark.sql(f"""
        INSERT INTO {metadata_table} (
            run_id,
            run_timestamp,
            rows_inserted,
            status,
            lookback_days,
            rows_processed,
            unique_locations,
            unique_dates,
            force_full_refresh,
            processing_time_seconds,
            error_message
        )
        VALUES (
            '{run_id}',
            TIMESTAMP'{run_timestamp}',
            {rows_processed},
            '{status}',
            {lookback_days},
            {rows_processed},
            {unique_locations},
            {unique_dates},
            {force_full_refresh},
            {processing_time:.2f},
            {'NULL' if error_message is None else f"'{error_message}'"}
        )
    """)

In [0]:
%sql
-- Summary statistics
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT location_sk) AS unique_locations,
  COUNT(DISTINCT trend_date_sk) AS unique_dates,
  MIN(trend_date_sk) AS earliest_date,
  MAX(trend_date_sk) AS latest_date,
  SUM(total_jobs) AS total_jobs,
  SUM(remote_jobs) AS total_remote_jobs,
  SUM(onsite_jobs) AS total_onsite_jobs,
  AVG(avg_salary_usd) AS overall_avg_salary,
  MAX(updated_at) AS last_refreshed
FROM workspace.gold.gold_location_trends;

In [0]:
%sql
-- Check for data quality issues
SELECT
  'Null location_sk' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_location_trends
WHERE location_sk IS NULL

UNION ALL

SELECT
  'Null trend_date_sk' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_location_trends
WHERE trend_date_sk IS NULL

UNION ALL

SELECT
  'Negative job counts' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_location_trends
WHERE total_jobs < 0 
   OR remote_jobs < 0 
   OR onsite_jobs < 0

UNION ALL

SELECT
  'Invalid avg_salary' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_location_trends
WHERE avg_salary_usd < 0 OR avg_salary_usd > 10000000

ORDER BY issue_count DESC;

In [0]:
%sql
-- Top 10 locations by total jobs
SELECT
  l.location_sk,
  l.city,
  l.state_province,
  l.country,
  SUM(lt.total_jobs) AS total_jobs,
  SUM(lt.remote_jobs) AS total_remote_jobs,
  AVG(lt.avg_salary_usd) AS avg_salary,
  COUNT(DISTINCT lt.trend_date_sk) AS days_active
FROM workspace.gold.gold_location_trends lt
LEFT JOIN workspace.warehouse.dim_location l ON lt.location_sk = l.location_sk
GROUP BY l.location_sk, l.city, l.state_province, l.country
ORDER BY total_jobs DESC
LIMIT 10;

In [0]:
%sql
-- Last 30 days: remote vs onsite breakdown
SELECT
  trend_date_sk,
  COUNT(DISTINCT location_sk) AS active_locations,
  SUM(total_jobs) AS daily_total_jobs,
  SUM(remote_jobs) AS daily_remote_jobs,
  SUM(onsite_jobs) AS daily_onsite_jobs,
  ROUND(100.0 * SUM(remote_jobs) / NULLIF(SUM(total_jobs), 0), 2) AS remote_pct,
  AVG(avg_salary_usd) AS avg_salary
FROM workspace.gold.gold_location_trends
WHERE trend_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 30), 'yyyyMMdd') AS INT)
GROUP BY trend_date_sk
ORDER BY trend_date_sk DESC;

In [0]:
%sql
-- View last 10 refresh runs
SELECT 
  run_id,
  run_timestamp,
  lookback_days,
  rows_processed,
  unique_locations,
  unique_dates,
  processing_time_seconds,
  status,
  error_message
FROM workspace.metadata.gold_location_trends_refresh_log
ORDER BY run_timestamp DESC
LIMIT 10;